<a href="https://colab.research.google.com/github/Santiago-Echeverri-Arteaga/Fisica_Computacional_2/blob/master/curso_2026_2/05_generativos/50_autoencoders.ipynb" target="_parent">
  <img src="https://colab.research.google.com/assets/colab-badge.svg"
       alt="Abrir en Colab"/>
</a>

# Autoencoders

**Pregunta guía:** ¿Qué representación comprimida conserva la información útil?<br>
**Duración sugerida:** 4 horas.<br>
**Entorno:** CPU; datos incluidos o generados en memoria.

El orden de trabajo es siempre: problema → matemática → implementación
mínima → biblioteca → evaluación → interpretación física.


**Requiere TensorFlow.** Un encoder $z=f_\theta(x)$ y un decoder
$\hat x=g_\phi(z)$ minimizan reconstrucción. Un cuello de botella fuerza
compresión, pero no garantiza variables físicamente interpretables. La
detección de anomalías presupone que el entrenamiento representa la
normalidad.


In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import tensorflow as tf
from sklearn.datasets import load_digits
from sklearn.model_selection import train_test_split
from tensorflow import keras
from tensorflow.keras import layers

keras.utils.set_random_seed(42)
X=load_digits().data.astype("float32")/16.0
X_dev,X_test=train_test_split(X,test_size=.2,random_state=42)
X_train,X_val=train_test_split(X_dev,test_size=.2,random_state=42)
entrada=keras.Input((64,))
z=layers.Dense(32,activation="relu")(entrada); z=layers.Dense(8,name="latente")(z)
x=layers.Dense(32,activation="relu")(z); salida=layers.Dense(64,activation="sigmoid")(x)
autoencoder=keras.Model(entrada,salida)
encoder=keras.Model(entrada,z)
autoencoder.compile(optimizer="adam",loss="mse")
historia=autoencoder.fit(X_train,X_train,validation_data=(X_val,X_val),epochs=80,batch_size=64,verbose=0,
                         callbacks=[keras.callbacks.EarlyStopping(patience=8,restore_best_weights=True)])
print("dimensión",X.shape[1],"→",encoder.output_shape[-1],"| test MSE",autoencoder.evaluate(X_test,X_test,verbose=0))


In [ ]:
reconstruidas=autoencoder.predict(X_test[:10],verbose=0)
fig,axes=plt.subplots(2,10,figsize=(14,3))
for i in range(10):
    axes[0,i].imshow(X_test[i].reshape(8,8),cmap="gray"); axes[1,i].imshow(reconstruidas[i].reshape(8,8),cmap="gray")
    axes[0,i].axis("off"); axes[1,i].axis("off")
axes[0,0].set_ylabel("original"); axes[1,0].set_ylabel("reconstrucción")
plt.show()

rng=np.random.default_rng(42)
anomalías=np.clip(X_test+rng.normal(0,.35,X_test.shape),0,1)
err_normal=np.mean((X_test-autoencoder.predict(X_test,verbose=0))**2,axis=1)
err_anómalo=np.mean((anomalías-autoencoder.predict(anomalías,verbose=0))**2,axis=1)
plt.hist(err_normal,alpha=.6,label="normal"); plt.hist(err_anómalo,alpha=.6,label="perturbado"); plt.legend(); plt.xlabel("error"); plt.show()


**Ejercicios:** compare dimensión latente 2, 8 y 32; visualice el espacio
2D; entrene denoising autoencoder; establezca un umbral sólo con
validación; explique por qué una anomalía bien reconstruida puede escapar.
